# Bronze Zone Data Discovery and Profiling

## High-Level Findings and System Overview

This notebook documents the exploratory analysis of the three raw Bronze data sources for the Quantum Data Lake platform. Before implementing pipeline stages, this discovery establishes the physical grain, structural invariants, schemas, and data engineering constraints of each source.

### Core Architectural Principles

1. **Simulated Syndromes (`qec_syndromes`):**
   - **Grain:** Pre-aggregated frequency counts. 75,598 rows represent 70,000,000 physical shots.
   - **Constraint:** Rows must remain aggregated in storage. All downstream statistical metrics and model fitting must use `quantity` as sample weights.
   - **Key Finding:** Syndromes are degenerate; identical syndrome bit patterns occur under both logical error labels (0 and 1). A syndrome string alone cannot serve as a unique primary key.

2. **Google Hardware Experiments (`google_qec`):**
   - **Grain:** One row per physical shot (50,000 shots per experiment across 25 QEC rounds).
   - **Format:** Stim binary little-endian bit-packed arrays (`.b8`) requiring strict byte alignment.
   - **Key Finding:** Distance 3 ($d=3$, 200 detectors) and Distance 5 ($d=5$, 600 detectors) have incompatible dimensions and must never be merged into flat feature matrices. Experiments vary by physical chip center coordinates to capture device heterogeneity. An actual logical flip is physical hardware damage, whereas a decoder error is an algorithmic misclassification (`prediction != actual`).

3. **Quantum Circuits (`qasmbench`):**
   - **Grain:** Static OpenQASM 2.0 program definitions of parity check extraction and conditional recovery.
   - **Constraint:** Circuits represent structural relational graphs, not runtime event traces. They share no run identifiers or shot indices with the other sources and cannot be joined row-by-row.


In [ ]:
import sys
from quantum_lake_student.config import Settings
from quantum_lake_student.connections import check_platform

print(f"Python Version in Container: {sys.version}")

# Verify connection to MinIO object store and PostgreSQL database
settings = Settings.from_environment()
status = check_platform(settings)
for service, msg in status.items():
    print(f"[OK] {service}: {msg}")


## 1. Source 1: Simulated Surface-Code Syndromes (`qec_syndromes`)

### Key Findings

- **Aggregation and Physical Grain:**
  The dataset authors executed simulated Monte Carlo shots, grouped identical observations by `(labels, syndromes)`, and recorded the frequency count in `quantity`:
  $$\text{GROUP BY (labels, syndromes)} \implies \text{COUNT}(*) \longrightarrow \mathbf{quantity}$$
  A row represents a distinct combination of syndrome and outcome, not an individual shot.

- **Physical Fault Rate (PFR) Scaling:**
  The 7 CSV archives scale from $p = 0.000010$ (0.001% gate noise) to $p = 0.010000$ (1.0% gate noise). At low error rates, almost all 10 million shots observe zero errors, collapsing into a 4.4 KB file. At high error rates, diverse error chains yield thousands of distinct syndrome patterns, expanding the file to 3.1 MB.

- **Degeneracy:**
  Because different combinations of physical faults can produce identical boundary detections, the same syndrome pattern occurs under both `labels = 0` and `labels = 1`. Syndromes cannot serve as unique primary keys.

### Pipeline Constraints
1. **Never expand rows:** Keep the data aggregated to avoid out-of-memory errors.
2. **Weighted computation:** All downstream evaluation metrics must compute frequency-weighted averages:
   $$\text{Logical Error Rate} = \frac{\sum (\text{labels} \times \text{quantity})}{\sum \text{quantity}}$$
3. **Model splits:** The train-validation-test split must isolate complete physical fault rate files to assess model generalization across noise regimes.


In [ ]:
import io
import zipfile
from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client

# 1. Initialize client and stream syndromes archive from MinIO
settings = Settings.from_environment()
client = minio_client(settings)

object_name = "bronze/source=qec_syndromes/syndromes_dataset.zip"
response = client.get_object(settings.s3_bucket, object_name)
zip_bytes = io.BytesIO(response.read())
response.close()
response.release_conn()

# 2. Inspect archive member files without extracting to disk
with zipfile.ZipFile(zip_bytes) as z:
    file_list = z.infolist()
    print(f"Total files in archive: {len(file_list)}\n")
    for file_info in file_list:
        print(f"  - {file_info.filename} ({file_info.file_size:,} bytes)")


In [ ]:
import pandas as pd

# Read the first 5 rows of a sample simulation file directly from the zip stream
with zipfile.ZipFile(zip_bytes) as z:
    with z.open("d-3_pfr-0.001000_nb-10M.csv") as f:
        df_sample = pd.read_csv(f, nrows=5)

print("Sample records (d-3_pfr-0.001000_nb-10M.csv):")
display(df_sample)


## 2. Source 2: Google Sycamore Hardware Experiments (`google_qec`)

### Key Findings

- **Physical Hardware Origin:**
  Data reflects actual superconducting circuits executed on Google Sycamore. There is no synthetic fault rate parameter; noise stems from environmental decoherence and physical gate imperfections.

- **Experiment Dimensions:**
  All 5 experiments share fixed execution invariants: 25 rounds and 50,000 shots in the X-basis.
  - Four experiments use Distance 3 ($d=3$): 9 data qubits, 8 measurement qubits, yielding $25 \times 8 = 200$ detector events per shot.
  - One experiment uses Distance 5 ($d=5$): 25 data qubits, 24 measurement qubits, yielding $25 \times 24 = 600$ detector events per shot.

- **Spatial Device Variation:**
  The four distance-3 experiments are centered at different coordinates on the processor chip (`(3,5)`, `(5,3)`, `(5,7)`, `(7,5)`). This captures spatial variance across physical qubits on the wafer.

- **Binary Alignment:**
  Stim `.b8` files store little-endian bit-packed records aligned to byte boundaries (for $d=3$, $\lceil 200/8 \rceil = 25$ bytes per shot). Text `.01` files store one binary decision per line across all 50,000 shots.

- **Ground Truth vs. Algorithmic Error:**
  `obs_flips_actual.01` is obtained via destructive final readout of all data qubits after round 25. An actual flip of 1 represents hardware memory failure. A decoder error occurs strictly when $\text{prediction} \neq \text{actual}$.

### Pipeline Constraints
1. **Never merge $d=3$ and $d=5$ raw detector arrays:** Their dimensions (200 vs. 600) and geometric neighborhoods are incompatible.
2. **Validate companion file consistency:** Verify that byte counts equal $\text{shots} \times \lceil \text{bits} / 8 \rceil$ across `.b8` files before constructing Silver tables.
3. **Multi-decoder ensembling:** Pre-computed decoder predictions (`pymatching`, `belief_matching`, `correlated_matching`) exhibit complementary failure modes, enabling downstream meta-decoders in Part II.


In [ ]:
import yaml
from quantum_lake_student.formats import iter_b8_records, parse_01_records

# 1. Fetch Google curated archive from MinIO
google_obj = client.get_object(
    settings.s3_bucket, "bronze/source=google_qec/google-surface-code-curated.zip"
)
google_zip = io.BytesIO(google_obj.read())
google_obj.close()
google_obj.release_conn()

# 2. Enumerate experiment directories and inspect metadata
with zipfile.ZipFile(google_zip) as z:
    all_files = z.namelist()
    experiments = sorted(list({f.split("/")[0] for f in all_files if "/" in f}))
    print("Experiments in Google release:")
    for exp in experiments:
        print(f"  - {exp}")

    first_exp = experiments[0]
    prop_content = z.read(f"{first_exp}/properties.yml").decode("utf-8")
    props = yaml.safe_load(prop_content)

    print(f"\nMetadata for {first_exp}:")
    print(f"  - Distance (d): {props['distance']}")
    print(f"  - Data Qubits: {props['data_qubits']}")
    print(f"  - Measurement Qubits: {props['measure_qubits']}")
    print(f"  - Rounds: {props['rounds']}")
    print(f"  - Total Shots: {props['shots']:,}")
    print(f"  - Detectors per Shot: {props['circuit_detectors']}")


In [ ]:
# Unpack sample detector events and compare decoder predictions with actual outcomes
with zipfile.ZipFile(google_zip) as z:
    detector_bytes = z.read(f"{first_exp}/detection_events.b8")
    actual_flips_raw = z.read(f"{first_exp}/obs_flips_actual.01")
    decoder_pred_raw = z.read(f"{first_exp}/obs_flips_predicted_by_pymatching.01")

detector_iterator = iter_b8_records(
    detector_bytes, bits_per_record=props["circuit_detectors"]
)
actual_flips = parse_01_records(actual_flips_raw)
pymatching_preds = parse_01_records(decoder_pred_raw)

print(f"Total shots parsed: {len(actual_flips):,}\n")

for shot_idx in range(3):
    detectors = next(detector_iterator)
    num_fired = sum(detectors)
    actual = actual_flips[shot_idx]
    pred = pymatching_preds[shot_idx]
    outcome = "MATCH (Correct Prediction)" if actual == pred else "MISMATCH (Decoder Error)"

    print(f"--- Shot #{shot_idx} ---")
    print(f"  Fired Detectors: {num_fired} / {props['circuit_detectors']}")
    print(f"  Predicted Observable Flip (PyMatching): {pred}")
    print(f"  Actual Observable Flip (Ground Truth):   {actual}")
    print(f"  Evaluation: {outcome}\n")


## 3. Source 3: Quantum Circuit Programs (`qasmbench`)

### Key Findings

- **Programmatic QEC Logic:**
  Files are written in OpenQASM 2.0. They define qubit allocation (`qreg q[3]`), ancilla allocation (`qreg a[2]`), classical syndrome registers (`creg syn[2]`), entangling parity checks (`cx`), and conditional recovery operations (`if(syn==1) x q[0]`).

- **Static Relational Graphs:**
  The circuits describe dependency topologies (which ancillas check which data qubits and how syndrome bits trigger corrections). They do not represent dynamic execution logs.

- **Independent Provenance:**
  These benchmark circuits do not contain shot instances or experiment IDs and cannot be joined row-by-row to Google hardware shots or simulation rows.

### Pipeline Constraints
1. **Do not simulate circuits:** The pipeline must parse static AST/text representations (qubit counts, gate counts, stabilizer connectivity), not simulate state vectors.
2. **Strict relational separation:** Store circuits in the common PostgreSQL database, but maintain them as isolated relational entities without artificial cross-source joins.


In [ ]:
# 1. Fetch QASMBench archive from MinIO
qasm_obj = client.get_object(
    settings.s3_bucket, "bronze/source=qasmbench/qasmbench-qec.zip"
)
qasm_zip = io.BytesIO(qasm_obj.read())
qasm_obj.close()
qasm_obj.release_conn()

# 2. Enumerate circuit files and inspect sample textbook implementation
with zipfile.ZipFile(qasm_zip) as z:
    all_files = sorted(z.namelist())
    qasm_circuits = [f for f in all_files if f.endswith(".qasm")]
    print("Quantum circuit files in QASMBench release:")
    for c in qasm_circuits:
        print(f"  - {c}")

    sample_qasm = (
        z.read("small/qec_sm_n5/qec_sm_n5.qasm")
        .decode("utf-8")
        .strip()
    )
    print("\nCircuit Code (small/qec_sm_n5/qec_sm_n5.qasm):\n")
    print(sample_qasm)
